# TP1 — Teste de Performance: Heap Binária
## Estruturas de Dados e Algoritmos Avançados

**Aluno:** João Victor Martins

**Organização do notebook**
- Cada exercício começa com o enunciado, seguido do código, da execução e da análise.
- A classe `BinaryHeap` é criada no **Exercício 2** com o estado interno (a lista) e as funções auxiliares. Cada exercício seguinte **acrescenta à classe o método que pede**, com `BinaryHeap.metodo = funcao`. Isso mantém o código de cada tarefa logo abaixo do seu enunciado, e todos os métodos continuam pertencendo à mesma classe e operando sobre `self`.
- A versão consolidada da classe, com todos os métodos em um único lugar, está no arquivo `binary_heap.py`, entregue junto com este notebook.
- Todas as comparações e trocas passam por dois métodos instrumentados (`_greater` e `_swap`). Assim, os números apresentados nas análises são contagens reais da execução, e não estimativas.

---
### Exercício 1: Motivação computacional para uso de Heaps

**Introdução**
Estruturas de dados não são escolhidas apenas por conveniência sintática, mas por propriedades formais que afetam diretamente desempenho, consumo de memória e previsibilidade de execução. Heaps binárias são amplamente utilizadas em sistemas que exigem priorização dinâmica, como escalonadores de tarefas, simuladores de eventos discretos e algoritmos de seleção parcial. Para um desenvolvedor de nível intermediário ou avançado, é essencial compreender em quais cenários o heap oferece vantagens claras em relação a listas ordenadas, filas simples ou árvores balanceadas. Este exercício estabelece a base conceitual necessária para as implementações práticas subsequentes, conectando requisitos computacionais reais com garantias assintóticas e estruturais fornecidas pela heap binária.

**Tarefa**
1. Defina um problema computacional real que envolva priorização dinâmica de elementos.
2. Justifique formalmente a escolha de uma heap binária para esse problema.
3. Compare a heap com pelo menos duas outras estruturas de dados possíveis.
4. Analise, em Big O, as operações críticas do problema proposto.

*Tarefa 1: Problema — fila de atendimento de um pronto-socorro com classificação de risco*

Em um pronto-socorro, os pacientes chegam continuamente e passam por uma triagem que atribui uma **prioridade** (por exemplo, no Protocolo de Manchester: emergência > muito urgente > urgente > pouco urgente > não urgente). Sempre que um médico fica livre, o sistema deve chamar o paciente **de maior prioridade** entre os que estão esperando. Em caso de empate, chama quem chegou primeiro.

A priorização é **dinâmica** por três motivos:
- novos pacientes entram a qualquer momento, inclusive com prioridade maior que a de todos os que já esperam;
- cada atendimento remove o paciente mais prioritário naquele instante;
- um paciente pode ser reclassificado se o seu quadro piorar.

O sistema precisa responder rápido mesmo com centenas de pacientes na fila em horário de pico. A prioridade pode ser codificada como uma tupla `(nível de risco, -ordem de chegada)`: a heap máxima devolve o maior nível e, no empate, quem chegou antes.

*Tarefa 2: Justificativa formal da escolha da heap binária*

As operações que o problema exige com frequência são:
- (a) inserir um paciente com sua prioridade;
- (b) consultar e remover o de maior prioridade;
- (c) consultar quem é o próximo sem removê-lo.

A heap binária máxima garante, por suas invariantes:
- **Ordem (I2):** todo pai é maior ou igual aos filhos, então o máximo está sempre na raiz. A consulta (c) custa $O(1)$.
- **Forma (I1):** é uma árvore binária quase completa, armazenada em um array sem buracos. Sua altura é $\lfloor \log_2 n \rfloor$, e tanto a inserção (sift-up, da folha até a raiz) quanto a remoção do máximo (sift-down, da raiz até a folha) percorrem no máximo um caminho dessa altura. Por isso, (a) e (b) custam $O(\log n)$ **no pior caso**, e não só em média.
- **Representação implícita:** os filhos e o pai são obtidos por aritmética de índices ($2i+1$, $2i+2$, $\lfloor (i-1)/2 \rfloor$). Não há ponteiros nem nós alocados separadamente, então a memória é $O(n)$ com constante mínima, e o acesso é contíguo, o que é bom para o cache.
- A heap **não mantém mais ordem do que o problema precisa**: ela não ordena a fila inteira, só garante quem é o próximo. Por isso é mais barata que uma estrutura totalmente ordenada.

*Tarefa 3: Comparação com outras estruturas*

| Estrutura | Inserir paciente | Chamar o próximo (remover máximo) | Ver o próximo | Observações |
|---|---|---|---|---|
| **Heap binária** | $O(\log n)$ | $O(\log n)$ | $O(1)$ | array simples, sem ponteiros; busca arbitrária é $O(n)$ |
| Lista não ordenada | $O(1)$ | $O(n)$: varre tudo para achar o máximo | $O(n)$ | cada atendimento fica lento quando a fila cresce |
| Lista ordenada (array) | $O(n)$: desloca elementos | $O(1)$: remove do fim | $O(1)$ | cada chegada fica lenta; mantém uma ordem total desnecessária |
| Fila FIFO simples | $O(1)$ | $O(1)$ | $O(1)$ | **não resolve o problema**: atende por ordem de chegada, ignorando o risco |
| Árvore balanceada (AVL, rubro-negra) | $O(\log n)$ | $O(\log n)$ | $O(\log n)$, ou $O(1)$ guardando o máximo | resolve, mas com 2–3 ponteiros por nó, rebalanceamentos e implementação bem mais complexa; só compensa se também for preciso buscar ou listar em ordem com frequência |

A lista ordenada e a não ordenada trocam um custo $O(n)$ de lugar, enquanto a heap equilibra as duas operações em $O(\log n)$. A árvore balanceada tem os mesmos limites, mas gasta mais memória e complexidade para oferecer recursos (busca e percurso ordenado) que este problema não usa.

*Tarefa 4: Análise em Big O das operações críticas*

Com $n$ = pacientes em espera e $m$ = eventos no dia (chegadas + atendimentos):

| Operação do sistema | Operação na heap | Custo |
|---|---|---|
| Paciente chega e é classificado | `insert` | $O(\log n)$ |
| Médico chama o próximo | `extract_max` | $O(\log n)$ |
| Painel mostra o próximo da fila | `peek` | $O(1)$ |
| Montar a fila de uma vez (por exemplo, migração de outro sistema) | `build_heap` | $O(n)$ |
| Reclassificar um paciente | `delete` + `insert` | $O(n)$ para localizá-lo + $O(\log n)$ |
| Um dia inteiro de operação | $m$ inserções e remoções | $O(m \log n)$ |

Ordem de grandeza: com $n = 1.000$ pacientes, cada chegada ou chamada custa no máximo $\lfloor \log_2 1000 \rfloor = 9$ trocas na heap. Uma lista ordenada deslocaria até 1.000 elementos por chegada, e uma não ordenada compararia 1.000 elementos por chamada.

O único ponto fraco é a reclassificação, que exige localizar o paciente: ela custa $O(n)$, porque a heap não serve para busca (o Exercício 5 mostra isso na prática). Se ela fosse frequente, a solução seria manter um dicionário `paciente → posição na heap`, atualizado a cada troca, o que a deixaria em $O(\log n)$.

---
### Exercício 2: Estruturação de Heap Binária em Python

**Introdução**
A implementação manual de uma heap binária exige domínio da relação entre estrutura lógica (árvore quase completa) e representação física em memória. Em ambientes profissionais, compreender essa relação é essencial para otimização, depuração e adaptação de bibliotecas de baixo nível. Neste exercício, o aluno deve estruturar corretamente uma heap binária máxima utilizando apenas arrays, garantindo que as propriedades fundamentais da estrutura sejam respeitadas desde a inicialização.

**Tarefa**
1. Implemente uma classe `BinaryHeap` em Python.
2. Utilize exclusivamente uma lista como estrutura interna de armazenamento.
3. Defina claramente as funções auxiliares de índice de pai e filhos.
4. Documente as invariantes que caracterizam uma heap binária válida.

In [1]:
class BinaryHeap:
    """Heap binária MÁXIMA implementada sobre uma única lista Python.

    Representação (árvore quase completa armazenada por níveis, da esquerda para a direita):
        raiz          -> índice 0
        pai de i      -> (i - 1) // 2
        filho esq. i  -> 2 * i + 1
        filho dir. i  -> 2 * i + 2

    Invariantes de uma heap binária máxima válida:
      (I1) Forma: os n elementos ocupam as posições 0..n-1 da lista, sem buracos.
           Isso equivale a uma árvore binária quase completa: todos os níveis cheios,
           exceto talvez o último, preenchido da esquerda para a direita.
      (I2) Ordem: para todo i > 0, data[pai(i)] >= data[i].
           Consequência: o maior elemento está sempre na raiz (data[0]).
      (I3) Altura: uma árvore quase completa com n nós tem altura floor(log2 n),
           o que limita o custo de sift-up e sift-down a O(log n).
    """

    def __init__(self, trace=False):
        self.data = []            # ÚNICA estrutura de armazenamento: uma lista
        # --- instrumentação (não guarda elementos da heap; usada nos Exercícios 3, 4, 9 e 11) ---
        self.trace = trace        # se True, cada troca é registrada em swap_log
        self.swap_log = []
        self.comparisons = 0      # comparações entre elementos
        self.swaps = 0            # trocas de posição

    # ---------- funções auxiliares de índice ----------
    @staticmethod
    def parent(i):
        """Índice do pai de i: (i - 1) // 2 — O(1)."""
        if i <= 0:
            raise IndexError("a raiz (índice 0) não tem pai")
        return (i - 1) // 2

    @staticmethod
    def left(i):
        """Índice do filho esquerdo de i: 2i + 1 — O(1)."""
        return 2 * i + 1

    @staticmethod
    def right(i):
        """Índice do filho direito de i: 2i + 2 — O(1)."""
        return 2 * i + 2

    # ---------- utilidades ----------
    def size(self):
        """Número de elementos — O(1)."""
        return len(self.data)

    def __len__(self):
        """Permite usar len(heap) — O(1)."""
        return len(self.data)

    def is_empty(self):
        """True se a heap não tem elementos — O(1)."""
        return len(self.data) == 0

    def peek(self):
        """Maior elemento sem remover — O(1) pela invariante I2."""
        if not self.data:
            raise IndexError("peek em heap vazia")
        return self.data[0]

    def reset_stats(self):
        """Zera os contadores de comparações e trocas e o registro de trocas."""
        self.comparisons = self.swaps = 0
        self.swap_log = []

    def _greater(self, a, b):
        """Toda comparação entre elementos passa por aqui, para ser contada."""
        self.comparisons += 1
        return a > b

    def _swap(self, i, j):
        """Troca duas posições; toda troca passa por aqui, para ser contada e registrada."""
        self.data[i], self.data[j] = self.data[j], self.data[i]
        self.swaps += 1
        if self.trace:
            self.swap_log.append((i, j, self.data[j], self.data[i]))   # (de, para, valor que subiu/desceu, outro)

    def check_invariants(self):
        """Verifica a invariante de ordem I2 em toda a lista (I1 é garantida pela própria lista)."""
        for i in range(1, len(self.data)):
            if self.data[(i - 1) // 2] < self.data[i]:
                return False
        return True

    def __repr__(self):
        return f"BinaryHeap({self.data})"


def show_tree(data, width=64):
    """Desenha a heap por níveis para visualizar a árvore representada pela lista."""
    if not data:
        print("(vazia)"); return
    level, start = 0, 0
    while start < len(data):
        count = 2 ** level
        nodes = data[start:start + count]
        slot = width // count
        print("".join(str(v).center(slot) for v in nodes).rstrip())
        start += count; level += 1


# Demonstração das funções de índice sobre uma heap válida montada à mão
h = BinaryHeap()
h.data = [90, 70, 80, 30, 60, 50, 10, 20]
print("lista interna:", h.data)
show_tree(h.data)
print(f"\n{'i':>3}{'valor':>7}{'pai(i)':>8}{'esq(i)':>8}{'dir(i)':>8}")
for i, v in enumerate(h.data):
    p = h.parent(i) if i > 0 else '-'
    l = h.left(i) if h.left(i) < len(h) else '-'
    r = h.right(i) if h.right(i) < len(h) else '-'
    print(f"{i:>3}{v:>7}{p:>8}{l:>8}{r:>8}")
print("\ninvariantes respeitadas?", h.check_invariants(), "| maior elemento (raiz):", h.peek())
h.data[3] = 95                                     # quebra a ordem: filho 95 > pai 70
print("após trocar data[3] por 95 -> invariantes respeitadas?", h.check_invariants())

lista interna: [90, 70, 80, 30, 60, 50, 10, 20]
                               90
               70                              80
       30              60              50              10
   20

  i  valor  pai(i)  esq(i)  dir(i)
  0     90       -       1       2
  1     70       0       3       4
  2     80       0       5       6
  3     30       1       7       -
  4     60       1       -       -
  5     50       2       -       -
  6     10       2       -       -
  7     20       3       -       -

invariantes respeitadas? True | maior elemento (raiz): 90
após trocar data[3] por 95 -> invariantes respeitadas? False


*Invariantes de uma heap binária máxima válida (Tarefa 4)*

| Invariante | Enunciado | Por que importa |
|---|---|---|
| **I1 — Forma** | os $n$ elementos ocupam as posições $0 \dots n-1$ da lista, sem buracos (árvore binária quase completa, preenchida por níveis da esquerda para a direita) | permite calcular pai e filhos por aritmética, sem ponteiros |
| **I2 — Ordem (heap máxima)** | para todo $i > 0$: `data[pai(i)] >= data[i]` | o máximo fica sempre na raiz, e `peek` custa $O(1)$ |
| **I3 — Altura** | consequência de I1: a altura é $\lfloor \log_2 n \rfloor$ | limita sift-up e sift-down, e portanto inserção e remoção, a $O(\log n)$ |

*Decisões e evidências*
- **Armazenamento (Tarefa 2):** os elementos ficam **somente** em `self.data`, uma `list` do Python. Os atributos `comparisons`, `swaps` e `swap_log` não guardam elementos da heap: são contadores de instrumentação, usados para registrar trocas (Ex. 3) e medir custos (Ex. 9 e 11).
- **Índices (Tarefa 3):** com a raiz no índice 0, temos `pai(i) = (i-1)//2`, `esq(i) = 2i+1` e `dir(i) = 2i+2`. Eles são métodos estáticos porque dependem só do índice, não do estado. `parent(0)` levanta `IndexError`, porque a raiz não tem pai.
- **Verificação:** a tabela de índices da saída mostra que cada posição aponta para o pai e os filhos corretos (por exemplo, o 20, no índice 7, tem como pai o 30, no índice 3). `check_invariants` aceita a heap montada à mão e rejeita a mesma lista depois que um filho (95) passa a ser maior que o pai (70).

---
### Exercício 3: Inserção em Heap com preservação de invariantes

**Introdução**
A operação de inserção em uma heap binária envolve mais do que simplesmente adicionar um elemento ao final da estrutura. O processo de reorganização, conhecido como *sift-up*, é fundamental para manter a propriedade de heap. Esse tipo de lógica aparece em sistemas críticos onde erros sutis podem comprometer todo o fluxo de execução. O domínio dessa operação é essencial para compreender o comportamento dinâmico da estrutura.

**Tarefa**
1. Implemente o método `insert(valor)` na classe `BinaryHeap`.
2. Registre cada troca realizada durante o processo de inserção.
3. Execute testes com sequências de entrada em ordens distintas.
4. Analise a complexidade temporal no pior caso.

In [2]:
import math, random

def _sift_up(self, i):
    """Sobe o elemento da posição i enquanto ele for MAIOR que o pai (restaura I2)."""
    while i > 0:
        p = (i - 1) // 2
        if self._greater(self.data[i], self.data[p]):
            self._swap(i, p)
            i = p
        else:
            break                     # pai >= filho: ordem restaurada, pode parar

def insert(self, valor):
    """Insere `valor` preservando as invariantes (append + sift-up) — O(log n).
    1) append no fim -> mantém a forma (I1), pois ocupa a próxima posição livre;
    2) sift-up -> restaura a ordem (I2) subindo pelo único caminho folha -> raiz."""
    self.data.append(valor)
    self._sift_up(len(self.data) - 1)

# Os métodos são acoplados à classe BinaryHeap definida no Exercício 2
BinaryHeap._sift_up = _sift_up
BinaryHeap.insert = insert


def demo_insercao(nome, valores):
    h = BinaryHeap(trace=True)
    print(f"=== Sequência {nome}: {valores}")
    for v in valores:
        antes = len(h.swap_log)
        h.insert(v)
        trocas = h.swap_log[antes:]
        desc = "; ".join(f"{a}↔{b} ({val} sobe)" for a, b, val, _ in trocas) or "nenhuma troca"
        print(f"  insert({v:>2}) trocas: {desc:<52} heap: {h.data}")
        assert h.check_invariants()
    print(f"  total: {h.swaps} trocas, {h.comparisons} comparações | válida: {h.check_invariants()}")
    show_tree(h.data, 48)
    print()
    return h

demo_insercao("crescente (pior caso: cada novo valor é o maior)", [10, 20, 30, 40, 50, 60, 70])
demo_insercao("decrescente (melhor caso: nunca sobe)", [70, 60, 50, 40, 30, 20, 10])
random.seed(3)
demo_insercao("aleatória", random.sample(range(1, 100), 7))
demo_insercao("com repetidos", [5, 8, 5, 8, 1, 8])

# Pior caso: o número de trocas de cada insert é limitado pela altura floor(log2 n)
print("Pior caso medido (entrada crescente) x limite teórico:")
print(f"{'n':>9}{'máx. trocas em um insert':>27}{'floor(log2 n)':>15}{'trocas totais':>15}{'n·log2 n':>12}")
for n in [10, 100, 1_000, 10_000, 100_000]:
    h = BinaryHeap(); pior = 0
    for v in range(n):
        antes = h.swaps; h.insert(v); pior = max(pior, h.swaps - antes)
    assert h.check_invariants()
    print(f"{n:>9}{pior:>27}{math.floor(math.log2(n)):>15}{h.swaps:>15}{round(n*math.log2(n)):>12}")

=== Sequência crescente (pior caso: cada novo valor é o maior): [10, 20, 30, 40, 50, 60, 70]
  insert(10) trocas: nenhuma troca                                        heap: [10]
  insert(20) trocas: 1↔0 (20 sobe)                                        heap: [20, 10]
  insert(30) trocas: 2↔0 (30 sobe)                                        heap: [30, 10, 20]
  insert(40) trocas: 3↔1 (40 sobe); 1↔0 (40 sobe)                         heap: [40, 30, 20, 10]
  insert(50) trocas: 4↔1 (50 sobe); 1↔0 (50 sobe)                         heap: [50, 40, 20, 10, 30]
  insert(60) trocas: 5↔2 (60 sobe); 2↔0 (60 sobe)                         heap: [60, 40, 50, 10, 30, 20]
  insert(70) trocas: 6↔2 (70 sobe); 2↔0 (70 sobe)                         heap: [70, 40, 60, 10, 30, 20, 50]
  total: 10 trocas, 10 comparações | válida: True
                       70
           40                      60
     10          30          20          50

=== Sequência decrescente (melhor caso: nunca sobe): [70, 60, 50, 40,

   100000                         16             16        1468946     1660964


*Análise da inserção*
- **Algoritmo:** o `append` coloca o novo valor na próxima posição livre, o que mantém a forma (I1). A seguir, o `sift-up` o compara com o pai e troca enquanto ele for maior, o que restaura a ordem (I2). Só o caminho entre a nova folha e a raiz pode ter sido afetado, porque o resto da árvore não muda.
- **Registro de trocas (Tarefa 2):** cada troca é registrada em `swap_log` como (posição de origem, posição de destino e valor que subiu) e aparece na saída de cada `insert`.
- **Ordens distintas (Tarefa 3):**
  - Crescente: cada novo valor é o maior de todos e sobe até a raiz (10 trocas para 7 elementos).
  - Decrescente: nenhum valor sobe (0 trocas, uma comparação por inserção).
  - Aleatória: algumas trocas (4).
  - Com repetidos: como a comparação é estrita (`>`), valores iguais ao pai não sobem, o que evita trocas desnecessárias.

  Em todos os casos, a heap foi validada após cada inserção.
- **Pior caso (Tarefa 4):** o elemento sobe no máximo até a raiz, ou seja, faz no máximo $h = \lfloor \log_2 n \rfloor$ trocas, com uma comparação por nível. Portanto, `insert` custa $O(\log n)$ no pior caso e $O(1)$ no melhor (quando o valor não é maior que o pai).
  - A tabela confirma exatamente o limite: a maior quantidade de trocas em um único `insert` foi 3, 6, 9, 13 e 16 para $n$ = 10, 100, 1.000, 10.000 e 100.000, igual a $\lfloor \log_2 n \rfloor$ em todos os casos.
  - Inserir $n$ elementos em ordem crescente custa $\sum_i \lfloor \log_2 i \rfloor = \Theta(n \log n)$: foram 1.468.946 trocas para $n = 100.000$, abaixo de $n \log_2 n \approx 1.660.964$.

---
### Exercício 4: Remoção do elemento raiz (extract-max)

**Introdução**
A remoção do elemento raiz é a operação central da heap binária. Ela exige uma reorganização completa da estrutura a partir da raiz, utilizando o processo de *sift-down*. Essa operação é crítica em filas de prioridade e algoritmos de seleção. Implementá-la corretamente demonstra domínio da lógica estrutural e da manutenção das invariantes da heap.

**Tarefa**
1. Implemente o método `extract_max()`.
2. Mostre passo a passo como a heap é reorganizada após cada remoção.
3. Execute a remoção sucessiva até a heap ficar vazia.
4. Analise a complexidade da operação em Big O.

In [3]:
def _sift_down(self, i):
    """Desce o elemento da posição i até que nenhum filho seja maior (restaura I2).
    A cada nível, troca com o MAIOR dos filhos, para que o novo pai domine os dois."""
    n = len(self.data)
    while True:
        l, r, maior = 2 * i + 1, 2 * i + 2, i
        if l < n and self._greater(self.data[l], self.data[maior]):
            maior = l
        if r < n and self._greater(self.data[r], self.data[maior]):
            maior = r
        if maior == i:
            break                     # nenhum filho é maior: posição correta
        self._swap(i, maior)
        i = maior

def extract_max(self):
    """Remove e devolve o máximo (a raiz) — O(log n).
    1) guarda a raiz;
    2) move o ÚLTIMO elemento para a raiz e encurta a lista -> mantém a forma (I1);
    3) sift-down a partir da raiz -> restaura a ordem (I2)."""
    if not self.data:
        raise IndexError("extract_max em heap vazia")
    maximo = self.data[0]
    ultimo = self.data.pop()
    if self.data:                     # se ainda há elementos, o último ocupa a raiz
        self.data[0] = ultimo
        self._sift_down(0)
    return maximo

BinaryHeap._sift_down = _sift_down
BinaryHeap.extract_max = extract_max


# Passo a passo: remoções sucessivas até a heap ficar vazia
h = BinaryHeap()
for v in [50, 30, 80, 10, 90, 60, 20, 70, 40]:
    h.insert(v)
print("Heap inicial:", h.data)
show_tree(h.data, 48)
h.trace = True
removidos = []
passo = 1
while not h.is_empty():
    h.swap_log = []
    raiz, ultimo = h.data[0], h.data[-1]
    m = h.extract_max()
    removidos.append(m)
    print(f"\nPasso {passo}: extract_max() -> {m}")
    if h.data:
        if h.swap_log:
            print(f"  o último elemento ({ultimo}) foi para a raiz e desceu (sift-down):",
                  "; ".join(f"posição {a}↔{b} ({val} desce)" for a, b, val, _ in h.swap_log))
        else:
            print(f"  o último elemento ({ultimo}) foi para a raiz e já está na posição correta")
    print(f"  heap agora: {h.data} | válida: {h.check_invariants()}")
    show_tree(h.data, 48)
    assert h.check_invariants()
    passo += 1
print("\nOrdem de remoção:", removidos)
assert removidos == sorted(removidos, reverse=True)
print("Os elementos saíram em ordem decrescente ✓")
try:
    h.extract_max()
except IndexError as e:
    print("extract_max() na heap vazia ->", e)

# Custo: trocas por remoção limitadas pela altura
print(f"\n{'n':>8}{'máx. trocas em um extract':>28}{'floor(log2 n)':>15}{'comparações totais':>20}{'2·n·log2 n':>12}")
for n in [100, 1_000, 10_000, 100_000]:
    h = BinaryHeap()
    for v in random.sample(range(n * 10), n):
        h.insert(v)
    h.reset_stats(); pior = 0
    while not h.is_empty():
        antes = h.swaps; h.extract_max(); pior = max(pior, h.swaps - antes)
    print(f"{n:>8}{pior:>28}{math.floor(math.log2(n)):>15}{h.comparisons:>20}{round(2*n*math.log2(n)):>12}")

Heap inicial: [90, 80, 60, 70, 30, 50, 20, 10, 40]
                       90
           80                      60
     70          30          50          20
  10    40

Passo 1: extract_max() -> 90
  o último elemento (40) foi para a raiz e desceu (sift-down): posição 0↔1 (40 desce); posição 1↔3 (40 desce)
  heap agora: [80, 70, 60, 40, 30, 50, 20, 10] | válida: True
                       80
           70                      60
     40          30          50          20
  10

Passo 2: extract_max() -> 80
  o último elemento (10) foi para a raiz e desceu (sift-down): posição 0↔1 (10 desce); posição 1↔3 (10 desce)
  heap agora: [70, 40, 60, 10, 30, 50, 20] | válida: True
                       70
           40                      60
     10          30          50          20

Passo 3: extract_max() -> 70
  o último elemento (20) foi para a raiz e desceu (sift-down): posição 0↔2 (20 desce); posição 2↔5 (20 desce)
  heap agora: [60, 40, 50, 10, 30, 20] | válida: True
               

  100000                          16             16             2831793     3321928


*Análise da remoção da raiz*
- **Algoritmo:**
  1. guarda a raiz (o máximo);
  2. move o **último** elemento para a raiz e encurta a lista. Remover qualquer outra posição abriria um buraco e violaria I1;
  3. faz o `sift-down`: compara o nó com os dois filhos e troca com o **maior** deles enquanto algum filho for maior. Trocar com o maior garante que o novo pai domine os dois filhos.

  `extract_max` em heap vazia levanta `IndexError`, em vez de devolver um valor ambíguo.
- **Passo a passo (Tarefas 2 e 3):** a saída mostra, a cada remoção, qual elemento foi para a raiz, por quais posições ele desceu e a árvore resultante, até a heap ficar vazia. Os elementos saíram em ordem decrescente (90, 80, …, 10), o que é a evidência de que a invariante foi mantida em todas as etapas. Esse processo é exatamente o núcleo do Heap Sort.
- **Complexidade (Tarefa 4):** o elemento desce no máximo $h = \lfloor \log_2 n \rfloor$ níveis, com 2 comparações e 1 troca por nível. Portanto, `extract_max` custa $O(\log n)$ no pior caso.
  - A tabela confirma: o máximo de trocas em uma remoção foi 6, 9, 13 e 16 para $n$ = 100, 1.000, 10.000 e 100.000, igual a $\lfloor \log_2 n \rfloor$.
  - Esvaziar a heap custa $O(n \log n)$: foram 2.831.793 comparações para $n = 100.000$, abaixo do limite $2 n \log_2 n \approx 3.321.928$.

---
### Exercício 5: Busca linear em Heap Binária

**Introdução**
Embora heaps não sejam projetadas para busca eficiente de elementos arbitrários, em aplicações reais essa funcionalidade pode ser necessária. Este exercício explora explicitamente essa limitação estrutural, forçando o aluno a lidar com as consequências algorítmicas dessa escolha de design.

**Tarefa**
1. Implemente um método `contains(valor)`.
2. Descreva o algoritmo utilizado para busca.
3. Analise a complexidade da operação.
4. Explique por que a heap não é adequada para buscas frequentes.

In [4]:
def contains(self, valor):
    """Busca linear: True se `valor` está na heap — O(n).
    Percorre a lista do índice 0 ao n-1 comparando cada elemento.
    A ordem da heap (I2) só relaciona pai e filhos; ela não diz em qual subárvore
    um valor arbitrário está, então não existe um 'lado certo' para descer como numa BST."""
    self.last_search_comparisons = 0
    for x in self.data:
        self.last_search_comparisons += 1
        if x == valor:
            return True
    return False

BinaryHeap.contains = contains

h = BinaryHeap()
for v in [50, 30, 80, 10, 90, 60, 20, 70, 40]:
    h.insert(v)
print("Heap:", h.data)
for alvo in [90, 40, 60, 55, 100]:
    achou = h.contains(alvo)
    print(f"contains({alvo:>3}) -> {str(achou):<5}  ({h.last_search_comparisons} comparações de {len(h)})")

# Custo em função de n: valor ausente (pior caso) e valor sorteado (caso médio)
print(f"\n{'n':>9}{'ausente: comparações':>22}{'presente (média de 200 buscas)':>32}{'n/2':>9}")
random.seed(5)
for n in [1_000, 10_000, 100_000]:
    h = BinaryHeap()
    valores = random.sample(range(n * 10), n)
    for v in valores:
        h.insert(v)
    h.contains(-1); ausente = h.last_search_comparisons
    total = 0
    for alvo in random.sample(valores, 200):
        assert h.contains(alvo); total += h.last_search_comparisons
    print(f"{n:>9}{ausente:>22}{total / 200:>32.1f}{n // 2:>9}")

# Por que a ordem da heap quase não ajuda: mesmo podando subárvores cuja raiz é menor
# que o alvo (lá embaixo só há valores ainda menores), buscar um valor pequeno visita quase tudo.
def contains_com_poda(heap, valor):
    visitados, pilha = 0, [0] if heap.data else []
    while pilha:
        i = pilha.pop(); visitados += 1
        if heap.data[i] == valor:
            return True, visitados
        if heap.data[i] > valor:                 # só vale descer se o nó ainda for maior que o alvo
            pilha += [c for c in (2 * i + 1, 2 * i + 2) if c < len(heap.data)]
    return False, visitados

h = BinaryHeap()
for v in random.sample(range(100_000), 10_000):
    h.insert(v)
ordenados = sorted(h.data)
print("\nBusca com poda (curiosidade) em uma heap de 10.000 elementos:")
for nome, alvo in [("valor grande (top 1%)", ordenados[-50]), ("valor mediano", ordenados[5_000]),
                   ("valor pequeno", ordenados[10]), ("ausente, menor que todos", -1)]:
    ok, vis = contains_com_poda(h, alvo)
    print(f"  {nome:<26} encontrado={str(ok):<5} nós visitados={vis:>6}")

Heap: [90, 80, 60, 70, 30, 50, 20, 10, 40]
contains( 90) -> True   (1 comparações de 9)
contains( 40) -> True   (9 comparações de 9)
contains( 60) -> True   (3 comparações de 9)
contains( 55) -> False  (9 comparações de 9)
contains(100) -> False  (9 comparações de 9)

        n  ausente: comparações  presente (média de 200 buscas)      n/2
     1000                  1000                           508.4      500
    10000                 10000                          5057.4     5000


   100000                100000                         51439.1    50000

Busca com poda (curiosidade) em uma heap de 10.000 elementos:
  valor grande (top 1%)      encontrado=True  nós visitados=    63
  valor mediano              encontrado=True  nós visitados=  6950
  valor pequeno              encontrado=True  nós visitados=  9347
  ausente, menor que todos   encontrado=False nós visitados= 10000


*Algoritmo, complexidade e limitação*
- **Algoritmo (Tarefa 2):** é uma busca linear. Percorre `self.data` do índice 0 ao $n-1$, comparando cada elemento com o valor procurado, e para no primeiro igual. As comparações são contadas em `last_search_comparisons`.
- **Complexidade (Tarefa 3):**
  - melhor caso $O(1)$ (o valor está na raiz: 1 comparação para o 90);
  - pior caso $O(n)$ (valor ausente: exatamente $n$ comparações, 1.000, 10.000 e 100.000 na tabela);
  - caso médio $\approx n/2$ para valores presentes (508, 5.057 e 51.439 contra $n/2$ = 500, 5.000 e 50.000). Ou seja, $O(n)$.
- **Por que a heap não serve para buscas frequentes (Tarefa 4):**
  - A ordem da heap é **parcial e só vertical**: ela garante pai ≥ filhos, mas não diz nada sobre irmãos nem sobre subárvores diferentes. Diferente de uma BST, não existe a regra "se for menor, vá para a esquerda", então não há como descartar metade da estrutura a cada passo.
  - A única poda possível é não descer em uma subárvore cuja raiz já é menor que o valor procurado. Ela ajuda apenas para valores grandes (63 nós visitados para um valor do top 1%), mas para valores medianos ou pequenos ainda visita quase tudo (6.950 e 9.347 de 10.000 nós), e para um valor ausente menor que todos visita os 10.000. O pior caso continua $O(n)$.
  - Se a aplicação precisasse buscar com frequência, o correto seria usar uma estrutura adequada (hashtable, com $O(1)$ esperado, ou árvore balanceada, com $O(\log n)$) ou combinar a heap com um índice auxiliar `valor → posição`, atualizado em cada troca.

---
### Exercício 6: Remoção de elemento arbitrário

**Introdução**
A remoção de elementos que não estão na raiz não é uma operação natural em heaps, mas pode ser necessária em sistemas avançados. Este exercício exige raciocínio cuidadoso para garantir que a estrutura permaneça válida após a remoção, reforçando a compreensão das invariantes da heap.

**Tarefa**
1. Implemente um método `delete(valor)`.
2. Reorganize corretamente a heap após a remoção.
3. Teste o método com valores existentes e inexistentes.
4. Analise o impacto da operação no desempenho global.

In [5]:
def delete(self, valor):
    """Remove uma ocorrência de `valor` (True) ou devolve False se não existe — O(n).
    1) localizar o índice i (busca linear, como no Exercício 5) -> O(n);
    2) mover o ÚLTIMO elemento para a posição i e encurtar a lista -> mantém a forma (I1);
    3) o elemento que chegou em i pode ser MAIOR que o novo pai (precisa subir)
       ou MENOR que algum filho (precisa descer). Aplicamos sift-up e, se ele não se
       mover, sift-down — no máximo um dos dois realmente troca algo -> O(log n)."""
    try:
        i = self.data.index(valor)       # busca linear
    except ValueError:
        return False
    ultimo = self.data.pop()
    if i < len(self.data):              # se o removido não era o próprio último
        self.data[i] = ultimo
        antes = self.swaps
        self._sift_up(i)
        if self.swaps == antes:         # não subiu: talvez precise descer
            self._sift_down(i)
    return True

BinaryHeap.delete = delete

def montar(valores):
    h = BinaryHeap()
    for v in valores:
        h.insert(v)
    return h

base = montar([100, 90, 50, 80, 70, 45, 40, 10, 20, 60, 65, 5, 4]).data
print("Heap base:", base)
show_tree(base, 64)
sobe = [100, 30, 90, 10, 20, 80, 85]        # heap válida em que o último elemento (85) é grande
casos = [
    ("raiz", base, 100),
    ("folha (o próprio último)", base, 4),
    ("nó interno — o substituto DESCE", base, 90),
    ("nó interno — o substituto SOBE", sobe, 10),
    ("inexistente", base, 999),
]
for nome, origem, alvo in casos:
    h = BinaryHeap(trace=True); h.data = list(origem)   # cópia de uma heap válida
    assert h.check_invariants()
    tamanho = len(h)
    ok = h.delete(alvo)
    movs = "; ".join(f"{a}↔{b} ({val} se move)" for a, b, val, _ in h.swap_log) or "nenhuma troca"
    print(f"\ndelete({alvo}) [{nome}] em {origem if origem is sobe else 'heap base'} -> {ok}")
    print(f"  trocas: {movs}")
    print(f"  heap: {h.data} | válida: {h.check_invariants()} | tamanho: {tamanho} -> {len(h)}")
    assert h.check_invariants() and len(h) == tamanho - (1 if ok else 0)

# Duplicatas: remove apenas uma ocorrência
h = montar([7, 7, 3, 7, 1])
print(f"\nHeap com duplicatas {h.data}: delete(7) -> {h.delete(7)} -> {h.data} (restaram {h.data.count(7)} setes)")

# Teste de estresse contra uma lista de referência
random.seed(6)
h, ref = BinaryHeap(), []
for _ in range(20_000):
    op = random.random()
    if op < 0.5:
        v = random.randint(0, 300); h.insert(v); ref.append(v)
    elif op < 0.8:
        v = random.randint(0, 300)
        assert h.delete(v) == (v in ref)
        if v in ref: ref.remove(v)
    elif ref:
        m = h.extract_max(); assert m == max(ref); ref.remove(m)
    assert h.check_invariants() and sorted(h.data) == sorted(ref)
print("Estresse: 20.000 operações aleatórias (insert, delete existente/inexistente, extract_max)"
      " -> heap válida e com o mesmo conteúdo da referência em todos os passos ✓")

# Impacto no desempenho: custo de delete dominado pela busca linear
print(f"\n{'n':>9}{'comparações da busca':>23}{'trocas para reorganizar':>26}{'log2 n':>9}")
for n in [1_000, 10_000, 100_000]:
    h = BinaryHeap()
    vals = random.sample(range(n * 10), n)
    for v in vals:
        h.insert(v)
    alvos = random.sample(vals, 100)
    busca = trocas = 0
    for a in alvos:
        busca += h.data.index(a) + 1                 # posições percorridas pela busca linear
        antes = h.swaps; h.delete(a); trocas += h.swaps - antes
    print(f"{n:>9}{busca / 100:>23.0f}{trocas / 100:>26.2f}{math.log2(n):>9.1f}")

Heap base: [100, 90, 50, 80, 70, 45, 40, 10, 20, 60, 65, 5, 4]
                              100
               90                              50
       80              70              45              40
   10      20      60      65      5       4

delete(100) [raiz] em heap base -> True
  trocas: 0↔1 (4 se move); 1↔3 (4 se move); 3↔8 (4 se move)
  heap: [90, 80, 50, 20, 70, 45, 40, 10, 4, 60, 65, 5] | válida: True | tamanho: 13 -> 12

delete(4) [folha (o próprio último)] em heap base -> True
  trocas: nenhuma troca
  heap: [100, 90, 50, 80, 70, 45, 40, 10, 20, 60, 65, 5] | válida: True | tamanho: 13 -> 12

delete(90) [nó interno — o substituto DESCE] em heap base -> True
  trocas: 1↔3 (4 se move); 3↔8 (4 se move)
  heap: [100, 80, 50, 20, 70, 45, 40, 10, 4, 60, 65, 5] | válida: True | tamanho: 13 -> 12

delete(10) [nó interno — o substituto SOBE] em [100, 30, 90, 10, 20, 80, 85] -> True
  trocas: 3↔1 (85 se move)
  heap: [100, 85, 90, 30, 20, 80] | válida: True | tamanho: 7 -> 6

de

Estresse: 20.000 operações aleatórias (insert, delete existente/inexistente, extract_max) -> heap válida e com o mesmo conteúdo da referência em todos os passos ✓

        n   comparações da busca   trocas para reorganizar   log2 n
     1000                    504                      0.81     10.0
    10000                   4655                      1.06     13.3


   100000                  51644                      0.83     16.6


*Reorganização após a remoção e impacto no desempenho*
- **Algoritmo (Tarefas 1 e 2):**
  1. localiza o índice $i$ do valor (busca linear);
  2. move o último elemento para $i$ e encurta a lista, o que mantém I1;
  3. restaura I2. O elemento que chegou em $i$ veio de outra subárvore, então pode ser **maior que o novo pai** (precisa subir) ou **menor que um filho** (precisa descer). Por isso aplicamos `sift-up` e, se ele não se moveu, `sift-down`. No máximo um dos dois faz trocas.
- **Testes (Tarefa 3):**
  - raiz (o substituto desce 3 níveis);
  - folha que é o próprio último elemento (só encurta a lista);
  - nó interno em que o substituto **desce**;
  - nó interno em que o substituto **sobe** (o 85 sobe para o lugar do 30);
  - valor inexistente (devolve `False` e não altera a heap);
  - duplicatas (remove só uma ocorrência).

  A heap foi validada em todos os casos, e um estresse de 20.000 operações aleatórias (inserções, deleções de valores existentes e inexistentes e `extract_max`) manteve a heap válida e com o mesmo conteúdo de uma lista de referência em todos os passos.
- **Impacto no desempenho global (Tarefa 4):** o custo é $O(n)$ para localizar mais $O(\log n)$ para reorganizar, ou seja, $O(n)$ no total.
  - As medições mostram que a busca domina completamente: em média 504, 4.655 e 51.644 posições percorridas para $n$ = 1.000, 10.000 e 100.000, contra só cerca de 1 troca para reorganizar.
  - Em um sistema que faz deleções arbitrárias com frequência, cada uma custa tanto quanto percorrer a heap inteira, o que anula a vantagem $O(\log n)$ das outras operações.
  - Há duas alternativas para esse cenário: um **índice auxiliar** `valor → posição` (heap indexada), que reduz `delete` a $O(\log n)$ ao custo de atualizar o índice em cada troca; ou a **remoção preguiçosa**, que marca o elemento como removido e o descarta só quando ele chegar à raiz.

---
### Exercício 7: Heap como array: validação estrutural

**Introdução**
A representação implícita da heap por meio de arrays é uma das principais razões de sua eficiência espacial. No entanto, essa representação exige disciplina rigorosa no cálculo de índices e na manutenção das relações hierárquicas. Este exercício avalia a capacidade do aluno de validar estruturalmente uma heap representada como array.

**Tarefa**
1. Implemente uma função `is_valid_heap(array)`.
2. Verifique se a propriedade de heap é respeitada em todo o array.
3. Teste a função com arrays válidos e inválidos.
4. Justifique o algoritmo adotado.

In [6]:
def is_valid_heap(array):
    """Devolve True se `array` representa uma heap binária MÁXIMA válida.
    Cada elemento, exceto a raiz, tem exatamente um pai: basta conferir a aresta
    pai -> filho de cada posição i >= 1, ou seja, n - 1 comparações no máximo.
    A forma (I1) não precisa ser verificada: numa lista não existem buracos."""
    for i in range(1, len(array)):
        if array[(i - 1) // 2] < array[i]:
            return False
    return True

def primeira_violacao(array):
    """Para diagnóstico: devolve (pai, filho) da primeira aresta que viola a ordem."""
    for i in range(1, len(array)):
        p = (i - 1) // 2
        if array[p] < array[i]:
            return (p, array[p]), (i, array[i])
    return None

testes = [
    ("vazia",                              [],                                 True),
    ("um elemento",                        [42],                               True),
    ("heap válida",                        [90, 70, 80, 30, 60, 50, 10, 20],   True),
    ("todos iguais",                       [5, 5, 5, 5, 5],                    True),
    ("ordenado decrescente",               [9, 8, 7, 6, 5, 4, 3],              True),
    ("ordenado crescente",                 [1, 2, 3, 4, 5],                    False),
    ("violação na raiz",                   [10, 20, 5],                        False),
    ("violação só no último nível",        [90, 70, 80, 30, 60, 50, 10, 75],   False),
    ("violação em nó interno profundo",    [90, 70, 80, 30, 60, 50, 10, 20, 35], False),
    ("heap MÍNIMA (não é máxima)",         [1, 3, 2, 7, 4],                    False),
]
for nome, arr, esperado in testes:
    r = is_valid_heap(arr)
    assert r == esperado, nome
    extra = "" if r else f" | primeira violação: pai {primeira_violacao(arr)[0]} < filho {primeira_violacao(arr)[1]}"
    print(f"{nome:<36} {str(arr):<36} -> {r}{extra}")

# Confere is_valid_heap contra as heaps produzidas pelos métodos da classe
random.seed(7)
for _ in range(200):
    h = BinaryHeap()
    for v in random.choices(range(50), k=random.randint(0, 60)):
        h.insert(v)
    assert is_valid_heap(h.data)
    if len(h) >= 2:
        corrompida = list(h.data); corrompida[-1] = corrompida[0] + 1   # folha maior que todos
        assert not is_valid_heap(corrompida)
print("\n200 heaps aleatórias geradas por insert: todas válidas; e todas detectadas após corrupção proposital ✓")

vazia                                []                                   -> True
um elemento                          [42]                                 -> True
heap válida                          [90, 70, 80, 30, 60, 50, 10, 20]     -> True
todos iguais                         [5, 5, 5, 5, 5]                      -> True
ordenado decrescente                 [9, 8, 7, 6, 5, 4, 3]                -> True
ordenado crescente                   [1, 2, 3, 4, 5]                      -> False | primeira violação: pai (0, 1) < filho (1, 2)
violação na raiz                     [10, 20, 5]                          -> False | primeira violação: pai (0, 10) < filho (1, 20)
violação só no último nível          [90, 70, 80, 30, 60, 50, 10, 75]     -> False | primeira violação: pai (3, 30) < filho (7, 75)
violação em nó interno profundo      [90, 70, 80, 30, 60, 50, 10, 20, 35] -> False | primeira violação: pai (3, 30) < filho (8, 35)
heap MÍNIMA (não é máxima)           [1, 3, 2, 7, 4]            

*Justificativa do algoritmo (Tarefa 4)*
- **Algoritmo:** para cada posição $i \ge 1$, verifica se `array[(i-1)//2] >= array[i]`. Ele para na primeira violação, o que é útil para diagnóstico: `primeira_violacao` mostra o par pai/filho que falhou.
- **Por que basta verificar pai e filho:** todo elemento, exceto a raiz, tem **exatamente um** pai, então as $n-1$ comparações cobrem todas as arestas da árvore, ou seja, o array inteiro (Tarefa 2). Se cada pai é maior ou igual aos seus filhos, por **transitividade** todo ancestral é maior ou igual a todos os seus descendentes. Não é preciso comparar avô com neto.
- **Por que a forma não precisa ser verificada:** em um array não existem buracos, então I1 vale automaticamente para qualquer lista. Só a ordem (I2) pode ser violada.
- **Custo:** $O(n)$ em tempo, com no máximo $n-1$ comparações, e $O(1)$ de memória extra. Não dá para fazer melhor no caso geral, porque qualquer elemento pode ser o que viola a ordem, e portanto todos precisam ser olhados.
- **Testes (Tarefa 3):**
  - válidos: vazio, um elemento, heap comum, todos iguais, ordem decrescente;
  - inválidos: ordem crescente, violação na raiz, violação só no último nível, violação em nó interno profundo e uma heap **mínima**, que é válida como mínima mas não como máxima.

  Além disso, 200 heaps geradas por `insert` foram todas aceitas, e todas foram rejeitadas depois de uma corrupção proposital.

---
### Exercício 8: Construção de Heap a partir de array não ordenado

**Introdução**
A construção eficiente de uma heap a partir de um array arbitrário é um problema clássico em ciência da computação. Diferentemente da inserção elemento a elemento, esse processo permite otimizações importantes. Este exercício reforça a compreensão da relação entre heap e algoritmos de ordenação parcial.

**Tarefa**
1. Implemente um método `build_heap(array)`.
2. Construa a heap sem utilizar inserções individuais.
3. Demonstre o funcionamento com arrays de tamanhos distintos.
4. Analise a complexidade total do algoritmo.

In [7]:
def build_heap(self, array):
    """Constrói a heap a partir de um array arbitrário em O(n), sem inserções individuais.
    Algoritmo de Floyd (bottom-up):
      1) copia o array para a lista interna -> a forma (I1) já está correta;
      2) as folhas (índices >= n//2) já são heaps de 1 elemento;
      3) do último nó interno (n//2 - 1) até a raiz, aplica sift-down:
         quando o nó i é processado, suas duas subárvores já são heaps,
         então um único sift-down transforma a subárvore de i em heap."""
    self.data = list(array)                      # cópia: não altera o array do chamador
    for i in range(len(self.data) // 2 - 1, -1, -1):
        self._sift_down(i)
    return self

BinaryHeap.build_heap = build_heap

# Passo a passo em um array pequeno
arr = [15, 42, 7, 99, 23, 4, 68, 31, 12, 50]
h = BinaryHeap(trace=True)
h.data = list(arr)
print("Array de entrada:", arr)
show_tree(h.data, 64)
for i in range(len(arr) // 2 - 1, -1, -1):
    h.swap_log = []
    v = h.data[i]
    h._sift_down(i)
    movs = "; ".join(f"{a}↔{b}" for a, b, _, _ in h.swap_log) or "já estava correto"
    print(f"sift_down(índice {i}, valor {v}): {movs:<22} -> {h.data}")
print("Heap final:"); show_tree(h.data, 64)
assert is_valid_heap(h.data)
assert h.data == BinaryHeap().build_heap(arr).data and arr == [15, 42, 7, 99, 23, 4, 68, 31, 12, 50]
print("build_heap(arr) produz o mesmo resultado, e o array original não foi alterado ✓\n")

# Arrays de tamanhos distintos (inclui casos de borda)
random.seed(8)
print(f"{'n':>8}{'nós internos processados':>26}{'trocas':>9}{'comparações':>13}{'trocas/n':>10}{'comparações/n':>15}  válida")
for n in [0, 1, 2, 3, 7, 10, 100, 1_000, 10_000, 100_000, 1_000_000]:
    entrada = list(range(n))                     # crescente: o pior caso para heap máxima
    h = BinaryHeap().build_heap(entrada)
    ok = is_valid_heap(h.data) and sorted(h.data) == entrada
    print(f"{n:>8}{n // 2:>26}{h.swaps:>9}{h.comparisons:>13}"
          f"{(h.swaps / n if n else 0):>10.3f}{(h.comparisons / n if n else 0):>15.3f}  {ok}")
    assert ok

# Também com entradas aleatórias e com repetidos
for nome, entrada in [("aleatória", random.sample(range(10**6), 10**5)),
                      ("muitos repetidos", random.choices(range(5), k=10**5))]:
    h = BinaryHeap().build_heap(entrada)
    assert is_valid_heap(h.data) and sorted(h.data) == sorted(entrada)
    print(f"entrada {nome:<17} n=100.000: trocas/n = {h.swaps / 10**5:.3f}, comparações/n = {h.comparisons / 10**5:.3f}, válida = True")

Array de entrada: [15, 42, 7, 99, 23, 4, 68, 31, 12, 50]
                               15
               42                              7
       99              23              4               68
   31      12      50
sift_down(índice 4, valor 23): 4↔9                    -> [15, 42, 7, 99, 50, 4, 68, 31, 12, 23]
sift_down(índice 3, valor 99): já estava correto      -> [15, 42, 7, 99, 50, 4, 68, 31, 12, 23]
sift_down(índice 2, valor 7): 2↔6                    -> [15, 42, 68, 99, 50, 4, 7, 31, 12, 23]
sift_down(índice 1, valor 42): 1↔3                    -> [15, 99, 68, 42, 50, 4, 7, 31, 12, 23]
sift_down(índice 0, valor 15): 0↔1; 1↔4; 4↔9          -> [99, 50, 68, 42, 23, 4, 7, 31, 12, 15]
Heap final:
                               99
               50                              68
       42              23              4               7
   31      12      15
build_heap(arr) produz o mesmo resultado, e o array original não foi alterado ✓

       n  nós internos processados   trocas  

 1000000                    500000   999988      1999974     1.000          2.000  True


entrada aleatória         n=100.000: trocas/n = 0.748, comparações/n = 1.884, válida = True
entrada muitos repetidos  n=100.000: trocas/n = 0.617, comparações/n = 1.738, válida = True


*Funcionamento e complexidade total*
- **Algoritmo (Tarefas 1 e 2):** é o algoritmo de Floyd, bottom-up. O array é copiado para a lista interna, e a forma (I1) já vale. As folhas (índices $\ge \lfloor n/2 \rfloor$) já são heaps de um elemento. Do último nó interno ($\lfloor n/2 \rfloor - 1$) até a raiz, aplica-se `sift-down`: quando o nó $i$ é processado, as duas subárvores dele já são heaps, e um sift-down basta para transformar a subárvore de $i$ em heap. Nenhuma chamada a `insert` é feita, e o array original não é alterado (a função trabalha sobre uma cópia).
- **Demonstração (Tarefa 3):** o passo a passo mostra os 5 `sift-down` de um array de 10 elementos. A tabela cobre $n$ = 0, 1, 2, 3, 7, 10, 100, até 1.000.000, além de entradas aleatórias e com muitos repetidos. Todas geraram heaps válidas com os mesmos elementos da entrada.
- **Complexidade total (Tarefa 4):** a análise ingênua daria $n/2$ sift-downs × $O(\log n)$ = $O(n \log n)$, mas esse limite é folgado. A maioria dos nós está perto das folhas e desce pouco:
  - há no máximo $\lceil n / 2^{h+1} \rceil$ nós de altura $h$, e cada um desce no máximo $h$ níveis;
  - o total é $\sum_{h \ge 0} \frac{n}{2^{h+1}} \cdot h = \frac{n}{2}\sum_{h\ge0}\frac{h}{2^h} = \frac{n}{2}\cdot 2 = n$;
  - portanto, **`build_heap` é $O(n)$**.
- **Evidência:** na entrada crescente, que é o pior caso para uma heap máxima, as trocas convergem para exatamente $1{,}000 \cdot n$ e as comparações para $2{,}000 \cdot n$ (999.988 trocas e 1.999.974 comparações para $n = 1.000.000$). Na entrada aleatória, foram $0{,}748 \cdot n$ trocas e $1{,}884 \cdot n$ comparações. A razão por $n$ fica **constante** com o crescimento, o que é a assinatura de $O(n)$.

---
### Exercício 9: Comparação entre build-heap e inserção incremental

**Introdução**
Decisões algorítmicas envolvem trade-offs claros entre simplicidade e eficiência. Comparar abordagens distintas para resolver o mesmo problema é uma habilidade essencial para engenheiros de computação. Este exercício explora essa comparação de forma prática e analítica.

**Tarefa**
1. Construa uma heap por inserção incremental.
2. Construa a mesma heap utilizando `build_heap`.
3. Compare o número de operações realizadas.
4. Discuta os resultados em termos de Big O.

In [8]:
import time

def por_insercao(array):
    h = BinaryHeap()
    for v in array:
        h.insert(v)
    return h

def por_build(array):
    return BinaryHeap().build_heap(array)

# 1-2) Mesma entrada, duas construções
entrada = [15, 42, 7, 99, 23, 4, 68, 31, 12, 50]
a, b = por_insercao(entrada), por_build(entrada)
print("Entrada:            ", entrada)
print("Inserção incremental:", a.data, f"| trocas={a.swaps}, comparações={a.comparisons}")
print("build_heap:          ", b.data, f"| trocas={b.swaps}, comparações={b.comparisons}")
print("Ambas válidas:", is_valid_heap(a.data) and is_valid_heap(b.data),
      "| mesmos elementos:", sorted(a.data) == sorted(b.data),
      "| mesma raiz (máximo):", a.peek() == b.peek())
print("Obs.: as listas internas podem diferir na ordem — existem várias heaps válidas para o mesmo conjunto.\n")

# 3) Número de operações em tamanhos crescentes e três padrões de entrada
def padrao(nome, n, rnd):
    if nome == "crescente":   return list(range(n))           # pior caso das duas para heap máxima
    if nome == "decrescente": return list(range(n, 0, -1))    # já é uma heap
    return rnd.sample(range(n * 10), n)

rnd = random.Random(9)
linhas = []
print(f"{'padrão':<12}{'n':>9}{'ops inserção':>15}{'ops build':>12}{'razão':>8}"
      f"{'tempo inserção':>16}{'tempo build':>13}   (ops = comparações + trocas)")
for nome in ["aleatório", "crescente", "decrescente"]:
    for n in [1_000, 10_000, 100_000, 1_000_000]:
        arr = padrao(nome, n, rnd)
        t0 = time.perf_counter(); hi = por_insercao(arr); ti = time.perf_counter() - t0
        t0 = time.perf_counter(); hb = por_build(arr);    tb = time.perf_counter() - t0
        assert is_valid_heap(hi.data) and is_valid_heap(hb.data) and hi.peek() == hb.peek()
        oi, ob = hi.comparisons + hi.swaps, hb.comparisons + hb.swaps
        linhas.append((nome, n, oi, ob))
        print(f"{nome:<12}{n:>9,}{oi:>15,}{ob:>12,}{oi / ob:>7.1f}x{ti:>15.2f}s{tb:>12.2f}s".replace(',', '.'))

# 4) Relação com Big O: o que fica CONSTANTE ao crescer n revela a classe de complexidade
print(f"\n{'padrão':<12}{'n':>11}{'inserção: ops/n':>17}{'inserção: ops/(n·log2 n)':>27}{'build: ops/n':>15}")
for nome, n, oi, ob in linhas:
    if nome != "decrescente":
        print(f"{nome:<12}{n:>11,}{oi / n:>17.3f}{oi / (n * math.log2(n)):>27.3f}{ob / n:>15.3f}".replace(',', '.'))

Entrada:             [15, 42, 7, 99, 23, 4, 68, 31, 12, 50]
Inserção incremental: [99, 50, 68, 31, 42, 4, 7, 15, 12, 23] | trocas=7, comparações=14
build_heap:           [99, 50, 68, 42, 23, 4, 7, 31, 12, 15] | trocas=6, comparações=14
Ambas válidas: True | mesmos elementos: True | mesma raiz (máximo): True
Obs.: as listas internas podem diferir na ordem — existem várias heaps válidas para o mesmo conjunto.

padrão              n   ops inserção   ops build   razão  tempo inserção  tempo build   (ops = comparações + trocas)
aleatório       1.000          3.495       2.605    1.3x           0.00s        0.00s
aleatório      10.000         35.124      25.951    1.4x           0.01s        0.01s


aleatório     100.000        356.106     262.653    1.4x           0.13s        0.09s


aleatório   1.000.000      3.561.631   2.624.146    1.4x           0.93s        0.73s
crescente       1.000         15.974       2.974    5.4x           0.01s        0.04s
crescente      10.000        227.262      29.974    7.6x           0.05s        0.01s


crescente     100.000      2.937.892     299.968    9.8x           0.53s        0.08s


crescente   1.000.000     35.902.890   2.999.962   12.0x           4.48s        0.62s
decrescente     1.000            999         999    1.0x           0.01s        0.03s
decrescente    10.000          9.999       9.999    1.0x           0.00s        0.00s
decrescente   100.000         99.999      99.999    1.0x           0.03s        0.02s


decrescente 1.000.000        999.999     999.999    1.0x           0.27s        0.24s

padrão                n  inserção: ops/n   inserção: ops/(n·log2 n)   build: ops/n
aleatório         1.000            3.495                      0.351          2.605
aleatório        10.000            3.512                      0.264          2.595
aleatório       100.000            3.561                      0.214          2.627
aleatório     1.000.000            3.562                      0.179          2.624
crescente         1.000           15.974                      1.603          2.974
crescente        10.000           22.726                      1.710          2.997
crescente       100.000           29.379                      1.769          3.000
crescente     1.000.000           35.903                      1.801          3.000


*Discussão em Big O (Tarefa 4)*
- **Mesma heap? (Tarefas 1 e 2):** as duas abordagens produzem heaps **válidas**, com os **mesmos elementos** e o **mesmo máximo**, mas listas internas que podem diferir na ordem. Isso é esperado, porque um mesmo conjunto admite várias organizações válidas.
- **Teoria:**
  - Inserção incremental: $n$ inserções, cada uma com custo $O(\log i)$, totalizando $\Theta(n \log n)$ no pior caso, porque a maioria dos nós está nos níveis de baixo e cada novo elemento pode subir até a raiz.
  - `build_heap`: $O(n)$, porque o trabalho pesado (descer muitos níveis) só acontece com os poucos nós perto da raiz.
- **Evidências (Tarefa 3):**
  - **Entrada crescente (pior caso):** a inserção teve `ops/(n·log₂ n)` estável em cerca de 1,6–1,8, ou seja, $\Theta(n \log n)$. O `build_heap` teve `ops/n` fixo em 3,0, ou seja, $\Theta(n)$. A vantagem do `build_heap` **cresce com $n$**: 5,4× com 1.000 elementos e 12,0× com 1.000.000 (35,9 milhões contra 3,0 milhões de operações, ≈5,4 s contra ≈0,7 s).
  - **Entrada aleatória:** as duas ficaram lineares na prática, com a inserção estável em cerca de 3,5 operações por elemento e o `build` em cerca de 2,6. Isso acontece porque, em média, um valor aleatório sobe apenas um número constante de níveis (o custo **médio** do `insert` é $O(1)$). Mesmo assim, o `build_heap` fez 1,4× menos operações. A diferença entre caso médio e pior caso é exatamente o que separa as duas linhas da tabela.
  - **Entrada decrescente:** o array já é uma heap, e as duas fazem apenas $n-1$ comparações, sem trocas.
- **Trade-off:** o `build_heap` é a escolha certa quando **todos os dados já estão disponíveis**, por exemplo para carregar uma fila ou iniciar um Heap Sort. A inserção incremental é necessária quando os elementos **chegam aos poucos**, como no pronto-socorro do Exercício 1, e custa $O(\log n)$ por chegada.

---
### Exercício 10: Heap sort parcial

**Introdução**
Heaps são frequentemente utilizadas em algoritmos de seleção, onde apenas parte dos dados precisa ser ordenada. Esse tipo de solução é comum em sistemas que trabalham com grandes volumes de dados e restrições de tempo.

**Tarefa**
1. Implemente um algoritmo que extraia os k maiores elementos de um array usando heap.
2. Não ordene completamente o array.
3. Teste o algoritmo com diferentes valores de k.
4. Analise a eficiência da solução.

In [9]:
def k_maiores(array, k):
    """Devolve os k maiores elementos de `array`, do maior para o menor, SEM ordenar o array todo.
    Estratégia A: build_heap em O(n) e depois k chamadas de extract_max, cada uma O(log n).
    Os n - k elementos restantes ficam na heap, sem nunca serem ordenados.
    Custo total: O(n + k log n)."""
    if not 0 <= k <= len(array):
        raise ValueError(f"k deve estar entre 0 e {len(array)}, recebido {k}")
    h = BinaryHeap().build_heap(array)
    resultado = [h.extract_max() for _ in range(k)]
    return resultado, h

def k_maiores_streaming(array, k):
    """Estratégia B (alternativa): mantém só os k maiores vistos até agora em uma heap de tamanho k.
    Precisamos remover o MENOR dos k a cada novo candidato, então usamos a mesma BinaryHeap
    máxima guardando valores NEGADOS (o máximo de -x é o mínimo de x).
    Custo O(n log k) e memória O(k): útil quando os dados chegam em fluxo ou não cabem na memória."""
    if not 0 <= k <= len(array):
        raise ValueError(f"k deve estar entre 0 e {len(array)}, recebido {k}")
    h = BinaryHeap()
    for x in array:
        if len(h) < k:
            h.insert(-x)
        elif k:
            h.comparisons += 1                 # compara o candidato com o menor dos k atuais
            if x > -h.peek():                  # x é maior: substitui o menor
                h.extract_max(); h.insert(-x)
    menores_primeiro = [-h.extract_max() for _ in range(len(h))]   # só os k escolhidos, O(k log k)
    return menores_primeiro[::-1], h

# Exemplo pequeno, mostrando que o restante NÃO foi ordenado
arr = [15, 42, 7, 99, 23, 4, 68, 31, 12, 50]
top3, resto = k_maiores(arr, 3)
print(f"array: {arr}\nk=3 -> {top3} | o que sobrou na heap (não ordenado): {resto.data}")
assert top3 == sorted(arr, reverse=True)[:3] and resto.data != sorted(resto.data, reverse=True)

# Diferentes valores de k, conferindo contra sorted() (usado só como gabarito)
random.seed(10)
n = 100_000
dados = random.sample(range(10 * n), n)
gabarito = sorted(dados, reverse=True)
print(f"\nn = {n:,}".replace(',', '.'))
print(f"{'k':>8}{'A: ops (build + k extract)':>28}{'n + k·log2 n':>15}{'B: ops (heap de k)':>21}{'n·log2 k':>12}  correto")
for k in [0, 1, 10, 100, 1_000, 10_000, n // 2, n]:
    ra, ha = k_maiores(dados, k)
    rb, hb = k_maiores_streaming(dados, k)
    ok = ra == rb == gabarito[:k]
    assert ok
    ops_a, ops_b = ha.comparisons + ha.swaps, hb.comparisons + hb.swaps
    teo_a = n + k * math.log2(n)
    teo_b = n * math.log2(k) if k > 1 else n
    print(f"{k:>8,}{ops_a:>28,}{round(teo_a):>15,}{ops_b:>21,}{round(teo_b):>12,}  {ok}".replace(',', '.'))

try:
    k_maiores([1, 2, 3], 5)
except ValueError as e:
    print("\nk_maiores([1, 2, 3], 5) ->", e)

array: [15, 42, 7, 99, 23, 4, 68, 31, 12, 50]
k=3 -> [99, 68, 50] | o que sobrou na heap (não ordenado): [42, 31, 15, 12, 23, 4, 7]

n = 100.000
       k  A: ops (build + k extract)   n + k·log2 n   B: ops (heap de k)    n·log2 k  correto


       0                     262.511        100.000                    0     100.000  True


       1                     262.558        100.017               99.999     100.000  True


      10                     262.980        100.166              100.881     332.193  True


     100                     267.190        101.661              114.882     664.386  True


   1.000                     309.181        116.610              250.352     996.578  True


  10.000                     727.074        266.096            1.297.622   1.328.771  True


  50.000                   2.528.528        930.482            3.704.693   1.560.964  True


 100.000                   4.494.876      1.760.964            4.588.586   1.660.964  True

k_maiores([1, 2, 3], 5) -> k deve estar entre 0 e 3, recebido 5


*Eficiência da solução (Tarefa 4)*
- **Estratégia A, `k_maiores` (Tarefas 1 e 2):** usa `build_heap` em $O(n)$ e depois $k$ chamadas de `extract_max`, cada uma $O(\log n)$. O custo total é **$O(n + k \log n)$**. Só os $k$ maiores saem ordenados; os $n-k$ restantes ficam na heap sem nunca serem ordenados, como mostra o exemplo (`[42, 31, 15, 12, 23, 4, 7]`). Só quando $k = n$ o algoritmo vira um Heap Sort completo, com $O(n \log n)$.
- **Estratégia B, `k_maiores_streaming` (alternativa):** mantém apenas os $k$ maiores vistos até agora em uma heap de tamanho $k$, usando a mesma `BinaryHeap` com valores negados, que funciona como uma heap mínima. O custo é $O(n \log k)$, com memória $O(k)$, e ela funciona com dados que chegam em fluxo ou não cabem na memória.
- **Testes com diferentes $k$ (Tarefa 3):** $k$ = 0, 1, 10, 100, 1.000, 10.000, 50.000 e 100.000, sobre $n = 100.000$. Os dois resultados coincidiram com o gabarito em todos os casos. `k` fora de $[0, n]$ levanta `ValueError`.
- **Leitura dos números:**
  - Na estratégia A, para $k$ pequeno o custo é dominado pelo `build_heap` (≈262 mil operações, ≈2,6·n). De $k = 0$ até $k = 1.000$, o custo cresce só de 262.511 para 309.181, porque cada extração acrescenta cerca de $3 \log_2 n$ operações.
  - Para $k$ bem pequeno, a estratégia B é ainda mais barata (100.881 operações para $k = 10$), porque a maior parte dos elementos é descartada com **uma única comparação** com o menor dos $k$ atuais.
  - Quando $k$ cresce, a B paga $O(\log k)$ para cada substituição e perde (1,30 milhão contra 0,73 milhão para $k = 10.000$).
  - Regra prática: para $k \ll n$, as duas evitam o $O(n \log n)$ de ordenar tudo. Use a B se os dados vierem em fluxo ou a memória for limitada, e a A se o array já estiver na memória e $k$ não for minúsculo.

---
### Exercício 11: Análise de complexidade empírica

**Introdução**
Além da análise teórica, a avaliação empírica do desempenho é fundamental em ambientes reais. Este exercício conecta teoria e prática, exigindo medições concretas de tempo e número de operações.

**Tarefa**
1. Instrumente o código da heap para contar operações.
2. Execute testes com tamanhos crescentes de entrada.
3. Registre os resultados obtidos.
4. Compare os dados empíricos com a análise teórica.

In [10]:
import pandas as pd

# 1) Instrumentação: todas as comparações passam por _greater e todas as trocas por _swap
#    (Exercício 2), e contains conta as próprias comparações (Exercício 5).
#    A função abaixo executa uma operação com os contadores zerados e mede também o tempo.
def medir(heap, operacao):
    heap.reset_stats()
    t0 = time.perf_counter()
    operacao()
    dt = time.perf_counter() - t0
    return heap.comparisons + heap.swaps, dt

# 2) Testes com tamanhos crescentes (dobrando n a cada passo)
rnd = random.Random(11)
registros = []
for n in [2 ** e for e in range(10, 18)]:            # 1.024 ... 131.072
    dados = rnd.sample(range(10 * n), n)

    h = BinaryHeap(); ops_ins, t_ins = medir(h, lambda: [h.insert(v) for v in dados])
    h2 = BinaryHeap(); ops_pior, t_pior = medir(h2, lambda: [h2.insert(v) for v in range(n)])
    hb = BinaryHeap(); ops_build, t_build = medir(hb, lambda: hb.build_heap(dados))

    # custo médio de UMA operação em uma heap que já tem n elementos (1.000 repetições)
    extras = range(10 * n, 10 * n + 1000)               # crescentes e maiores que todos: cada um sobe até a raiz
    ops_1ins, _ = medir(hb, lambda: [hb.insert(v) for v in extras])
    ops_1ext, _ = medir(hb, lambda: [hb.extract_max() for _ in range(1000)])
    hb.contains(-1); ops_busca = hb.last_search_comparisons

    he = BinaryHeap().build_heap(dados)
    ops_ext, t_ext = medir(he, lambda: [he.extract_max() for _ in range(n)])

    registros.append({'n': n,
        'insert n× (aleatório)': ops_ins, 'insert n× (pior caso)': ops_pior, 'build_heap': ops_build,
        'extract_max n×': ops_ext, '1 insert (pior caso)': ops_1ins / 1000, '1 extract_max': ops_1ext / 1000,
        'contains (ausente)': ops_busca,
        't insert n× (s)': t_ins, 't build (s)': t_build, 't extract n× (s)': t_ext})

# 3) Registro dos resultados
df = pd.DataFrame(registros)
pd.set_option('display.width', 200)
print("Operações contadas (comparações + trocas):")
print(df[['n', 'insert n× (aleatório)', 'insert n× (pior caso)', 'build_heap', 'extract_max n×',
          '1 insert (pior caso)', '1 extract_max', 'contains (ausente)']].to_string(index=False))
print("\nTempos (segundos):")
print(df[['n', 't insert n× (s)', 't build (s)', 't extract n× (s)']].round(4).to_string(index=False))
df.to_csv('resultados_empiricos_tp1.csv', index=False)
print("\nResultados salvos em resultados_empiricos_tp1.csv")

# 4) Comparação com a teoria: dividir pelas funções teóricas deve dar valores ~constantes
lg = df['n'].apply(math.log2)
teoria = pd.DataFrame({
    'n': df['n'],
    'build/n  [O(n)]': df['build_heap'] / df['n'],
    'insert pior/(n·lg n)  [O(n log n)]': df['insert n× (pior caso)'] / (df['n'] * lg),
    'extract n×/(n·lg n)  [O(n log n)]': df['extract_max n×'] / (df['n'] * lg),
    '1 insert pior/lg n  [O(log n)]': df['1 insert (pior caso)'] / lg,
    '1 extract/lg n  [O(log n)]': df['1 extract_max'] / lg,
    'contains/n  [O(n)]': df['contains (ausente)'] / df['n'],
    'insert aleatório/n  [O(1) médio]': df['insert n× (aleatório)'] / df['n'],
})
print("\nDados empíricos normalizados pela complexidade teórica (≈ constante = teoria confirmada):")
print(teoria.round(3).to_string(index=False))

# Fator de crescimento ao DOBRAR n (O(n) -> 2,0; O(n log n) -> um pouco acima de 2; O(log n) -> +1 operação/nível)
cresc = pd.DataFrame({
    'n': df['n'][1:].values,
    '1 insert pior (+ops)': (df['1 insert (pior caso)'] - df['1 insert (pior caso)'].shift())[1:].values,
    'build (×)': (df['build_heap'] / df['build_heap'].shift())[1:].values,
    'insert pior n× (×)': (df['insert n× (pior caso)'] / df['insert n× (pior caso)'].shift())[1:].values,
    'extract n× (×)': (df['extract_max n×'] / df['extract_max n×'].shift())[1:].values,
    '1 extract (+ops)': (df['1 extract_max'] - df['1 extract_max'].shift())[1:].values,
})
print("\nAo dobrar n:")
print(cresc.round(3).to_string(index=False))

Operações contadas (comparações + trocas):
     n  insert n× (aleatório)  insert n× (pior caso)  build_heap  extract_max n×  1 insert (pior caso)  1 extract_max  contains (ausente)
  1024                   3521                  16408        2651           23089                  20.0           27.5                1024
  2048                   7048                  36890        5348           52050                  22.0           30.5                2048
  4096                  14298                  81948       10667          116445                  24.0           33.5                4096
  8192                  29573                 180254       21635          257265                  26.0           36.5                8192
 16384                  58238                 393248       42911          563300                  28.0           39.5               16384
 32768                 116496                 852002       86042         1225525                  30.0           42.5            

*Comparação entre dados empíricos e teoria (Tarefa 4)*
- **Instrumentação (Tarefa 1):** toda comparação entre elementos passa por `_greater` e toda troca por `_swap`, que incrementam contadores da própria instância. A função `medir` zera os contadores, executa a operação e mede o tempo com `time.perf_counter`. Contar operações é mais confiável do que medir tempo, porque não depende da máquina nem de ruído do sistema.
- **Tamanhos crescentes (Tarefas 2 e 3):** $n$ dobra de 1.024 até 131.072. Os resultados ficam registrados nas tabelas acima e no arquivo `resultados_empiricos_tp1.csv`.
- **Como ler:** dividir as operações medidas pela função teórica deve dar um valor **aproximadamente constante**. Se a classe de complexidade estivesse errada, a razão cresceria ou cairia com $n$.

| Operação | Teoria | Empírico (razão normalizada) | Conclusão |
|---|---|---|---|
| `build_heap` | $O(n)$ | `ops/n` ≈ 2,59–2,64 em todos os tamanhos; ao dobrar $n$, as operações dobram (×1,98–2,03) | confirma $O(n)$ |
| 1 `insert` (pior caso) | $O(\log n)$ | exatamente $2\log_2 n$ (1 comparação + 1 troca por nível); **+2 operações a cada vez que $n$ dobra**, porque a árvore ganha um nível | confirma $O(\log n)$ |
| 1 `extract_max` | $O(\log n)$ | `ops/lg n` ≈ 2,7–2,9; **+3 operações a cada dobra** (2 comparações + 1 troca por nível novo) | confirma $O(\log n)$ |
| $n$ inserções (pior caso) | $O(n \log n)$ | `ops/(n lg n)` ≈ 1,6–1,8; ao dobrar $n$, ×2,14–2,25 (um pouco acima de 2, como esperado para $n \log n$) | confirma $\Theta(n \log n)$ |
| $n$ × `extract_max` | $O(n \log n)$ | `ops/(n lg n)` ≈ 2,3–2,6, com o mesmo crescimento ×2,15–2,25 | confirma $O(n \log n)$ |
| `contains` (ausente) | $O(n)$ | exatamente $1{,}0 \cdot n$ comparações | confirma $O(n)$ |
| $n$ inserções (aleatório) | $O(n \log n)$ no pior caso | `ops/n` ≈ 3,4–3,6, constante | o caso médio do `insert` é $O(1)$ |

- **Observações:**
  - As razões de `n lg n` crescem levemente porque os termos de ordem menor (por exemplo, $-n$ em $\sum \lfloor \log_2 i \rfloor = n\log_2 n - O(n)$) perdem peso com $n$ grande, e a razão tende a uma constante.
  - Os tempos seguem as mesmas tendências (o `extract` de $n$ elementos cresce mais que linearmente, enquanto o `build` acompanha $n$), mas com ruído, por isso a conclusão se baseia principalmente nas contagens.

---
### Exercício 12: Documentação técnica e justificativa de projeto

**Introdução**
Código sem documentação adequada é um passivo técnico. Em ambientes profissionais, a clareza das decisões de projeto é tão importante quanto a correção funcional. Este exercício avalia a capacidade do aluno de justificar tecnicamente suas escolhas e organizar o conhecimento produzido.

**Tarefa**
1. Documente a arquitetura da sua implementação de heap.
2. Justifique as principais decisões algorítmicas.
3. Relacione as decisões às propriedades formais da heap.
4. Prepare o material para apresentação técnica em vídeo.

*Tarefa 1: Arquitetura da implementação*

```
BinaryHeap
├── Estado interno
│   └── data : list            ← única estrutura de armazenamento (árvore quase completa por níveis)
├── Instrumentação (não armazena elementos)
│   ├── comparisons, swaps     ← contadores
│   └── trace, swap_log        ← registro opcional de cada troca
├── Índices (estáticos, O(1))  parent(i) · left(i) · right(i)
├── Primitivas privadas        _greater(a, b) · _swap(i, j) · _sift_up(i) · _sift_down(i)
├── Operações públicas         insert · extract_max · peek · contains · delete · build_heap
└── Utilidades                 size · __len__ · is_empty · check_invariants · reset_stats

Funções fora da classe:  is_valid_heap(array) · k_maiores(array, k) · k_maiores_streaming(array, k)
```

- **Camadas:** as operações públicas são escritas apenas em termos de duas primitivas (`_sift_up` e `_sift_down`), que por sua vez só alteram a lista por meio de `_swap`. Isso concentra a lógica que pode quebrar invariantes em poucos pontos e torna a instrumentação automática.
- **Encapsulamento:** o estado (`data`) só é alterado pelos métodos da classe, então qualquer sequência de chamadas públicas mantém a heap válida, como o teste integrado demonstra. `is_valid_heap` fica **fora** da classe porque o enunciado pede para validar *qualquer* array, inclusive um que não veio da classe.

*Tarefa 2: Principais decisões algorítmicas e justificativas*

| Decisão | Justificativa |
|---|---|
| Heap **máxima** com raiz no índice 0 | atende ao enunciado; índices `2i+1`, `2i+2` e `(i-1)//2` são simples e baratos |
| Comparação **estrita** (`>`) no sift-up e no sift-down | elementos iguais não trocam de lugar: menos trocas com duplicatas e mesmo resultado válido |
| No sift-down, trocar com o **maior** filho | garante que o novo pai seja maior que os dois filhos (trocar com o menor quebraria I2 no outro lado) |
| `extract_max` e `delete` usam o **último** elemento como substituto | é a única remoção que não abre buraco na lista (mantém I1) e custa $O(1)$ |
| `delete` aplica sift-up **ou** sift-down | o substituto vem de outra subárvore e pode precisar subir ou descer; nunca os dois |
| `build_heap` pelo método de **Floyd** (bottom-up) | $O(n)$ contra $O(n \log n)$ da inserção incremental (Ex. 8 e 9) |
| `contains` linear, documentado como $O(n)$ | a ordem da heap não permite descartar subárvores de forma geral (Ex. 5) |
| Erros explícitos (`IndexError`, `ValueError`) | operações em heap vazia ou com `k` inválido falham de forma clara, em vez de devolver valores ambíguos |
| `build_heap` copia o array de entrada | não altera os dados do chamador (efeito colateral surpreendente) |
| Contagem centralizada em `_greater` e `_swap` | os números das análises são contagens reais, sem instrumentação espalhada pelo código |

*Tarefa 3: Relação com as propriedades formais*

| Propriedade formal | Como cada operação a preserva ou explora | Consequência em custo |
|---|---|---|
| **I1 — Forma** (array sem buracos) | `insert` usa `append`; `extract_max` e `delete` usam o último elemento e `pop()`; `build_heap` copia o array inteiro | pai e filhos por aritmética; memória $O(n)$ sem ponteiros |
| **I2 — Ordem** (pai ≥ filhos) | `_sift_up` corrige o caminho folha → raiz; `_sift_down` corrige raiz → folha; `is_valid_heap` e `check_invariants` verificam aresta por aresta | máximo em `data[0]`: `peek` $O(1)$ |
| **I3 — Altura** $\lfloor \log_2 n \rfloor$ | os dois sifts percorrem no máximo um caminho entre raiz e folha | `insert`, `extract_max` e o reajuste do `delete` em $O(\log n)$, confirmados nos Ex. 3, 4 e 11 |
| Ordem **apenas parcial** | nada relaciona irmãos ou subárvores distintas | `contains` e a localização no `delete` em $O(n)$ (Ex. 5 e 6) |
| Maioria dos nós perto das folhas | no `build_heap`, quase todos os sift-downs são curtos | $\sum h \cdot n/2^{h+1} = O(n)$ (Ex. 8) |

| Operação | Complexidade | Exercício |
|---|---|---|
| `peek` | $O(1)$ | 2 |
| `insert` | $O(\log n)$ pior caso, $O(1)$ médio | 3, 9, 11 |
| `extract_max` | $O(\log n)$ | 4, 11 |
| `contains` | $O(n)$ | 5 |
| `delete` | $O(n)$ (busca) + $O(\log n)$ (reajuste) | 6 |
| `is_valid_heap` | $O(n)$ | 7 |
| `build_heap` | $O(n)$ | 8, 9, 11 |
| `k_maiores` / `k_maiores_streaming` | $O(n + k\log n)$ / $O(n \log k)$ | 10 |

In [11]:
import inspect

# 1) Arquitetura: a interface pública consolidada da classe BinaryHeap
print("class BinaryHeap  — heap binária MÁXIMA sobre uma lista (self.data)\n")
grupos = {
    "Índices (Ex. 2)":                  ["parent", "left", "right"],
    "Consulta / utilidades (Ex. 2)":    ["size", "__len__", "is_empty", "peek", "check_invariants"],
    "Inserção (Ex. 3)":                 ["insert", "_sift_up"],
    "Remoção da raiz (Ex. 4)":          ["extract_max", "_sift_down"],
    "Busca (Ex. 5)":                    ["contains"],
    "Remoção arbitrária (Ex. 6)":       ["delete"],
    "Construção em O(n) (Ex. 8)":       ["build_heap"],
    "Instrumentação (Ex. 3, 4, 9, 11)": ["_greater", "_swap", "reset_stats"],
}
for grupo, metodos in grupos.items():
    print(grupo)
    for m in metodos:
        f = getattr(BinaryHeap, m)
        doc = (inspect.getdoc(f) or "").splitlines()[0] if inspect.getdoc(f) else ""
        print(f"   {m}{inspect.signature(f)}  — {doc}")
print("Funções fora da classe")
for f in (is_valid_heap, k_maiores, k_maiores_streaming):
    print(f"   {f.__name__}{inspect.signature(f)}  — {inspect.getdoc(f).splitlines()[0]}")

# 2) Teste integrado final: todas as operações juntas, comparadas com uma referência simples
random.seed(12)
h, ref = BinaryHeap(), []
contagem = {'insert': 0, 'extract_max': 0, 'delete': 0, 'contains': 0, 'build_heap': 0}
for passo in range(30_000):
    r = random.random()
    if r < 0.40:
        v = random.randint(-500, 500); h.insert(v); ref.append(v); contagem['insert'] += 1
    elif r < 0.60 and ref:
        assert h.extract_max() == max(ref); ref.remove(max(ref)); contagem['extract_max'] += 1
    elif r < 0.80:
        v = random.randint(-500, 500); ok = h.delete(v)
        assert ok == (v in ref)
        if ok: ref.remove(v)
        contagem['delete'] += 1
    elif r < 0.995:
        v = random.randint(-500, 500); assert h.contains(v) == (v in ref); contagem['contains'] += 1
    else:
        novo = [random.randint(-500, 500) for _ in range(random.randint(0, 200))]
        h.build_heap(novo); ref = list(novo); contagem['build_heap'] += 1
    assert is_valid_heap(h.data) and len(h) == len(ref)
    if ref:
        assert h.peek() == max(ref)
print(f"\nTeste integrado: 30.000 operações {contagem}")
print("-> após CADA operação: is_valid_heap = True, tamanho correto e peek() == máximo da referência ✓")

class BinaryHeap  — heap binária MÁXIMA sobre uma lista (self.data)

Índices (Ex. 2)
   parent(i)  — Índice do pai de i: (i - 1) // 2 — O(1).
   left(i)  — Índice do filho esquerdo de i: 2i + 1 — O(1).
   right(i)  — Índice do filho direito de i: 2i + 2 — O(1).
Consulta / utilidades (Ex. 2)
   size(self)  — Número de elementos — O(1).
   __len__(self)  — Permite usar len(heap) — O(1).
   is_empty(self)  — True se a heap não tem elementos — O(1).
   peek(self)  — Maior elemento sem remover — O(1) pela invariante I2.
   check_invariants(self)  — Verifica a invariante de ordem I2 em toda a lista (I1 é garantida pela própria lista).
Inserção (Ex. 3)
   insert(self, valor)  — Insere `valor` preservando as invariantes (append + sift-up) — O(log n).
   _sift_up(self, i)  — Sobe o elemento da posição i enquanto ele for MAIOR que o pai (restaura I2).
Remoção da raiz (Ex. 4)
   extract_max(self)  — Remove e devolve o máximo (a raiz) — O(log n).
   _sift_down(self, i)  — Desce o elemento da posiç


Teste integrado: 30.000 operações {'insert': 12079, 'extract_max': 5983, 'delete': 5916, 'contains': 5872, 'build_heap': 150}
-> após CADA operação: is_valid_heap = True, tamanho correto e peek() == máximo da referência ✓


*Tarefa 4: Material para a apresentação técnica em vídeo (roteiro de 8 a 10 minutos)*

| Tempo | O que mostrar | O que dizer |
|---|---|---|
| 0:00–0:45 | Apresentação e Exercício 1 | o problema do pronto-socorro e por que a heap equilibra inserção e remoção em $O(\log n)$, contra $O(n)$ das listas |
| 0:45–2:00 | Exercício 2: a classe e a tabela de índices | uma única lista; `2i+1`, `2i+2` e `(i-1)//2`; as três invariantes |
| 2:00–3:30 | Exercícios 3 e 4, executando as células | o `swap_log` mostrando o valor subindo; o passo a passo do `extract_max` até esvaziar, com a saída em ordem decrescente; máximo de trocas = $\lfloor \log_2 n \rfloor$ |
| 3:30–4:45 | Exercícios 5 e 6 | por que a busca é $O(n)$ (ordem parcial); o `delete` em que o substituto sobe e o em que ele desce; a busca domina o custo |
| 4:45–5:30 | Exercício 7 | por que basta comparar cada nó com o pai (transitividade) |
| 5:30–7:00 | Exercícios 8 e 9 | o passo a passo do Floyd; a soma $\sum h \cdot n/2^{h+1} = n$; a tabela com 12× menos operações para 1 milhão de elementos crescentes |
| 7:00–8:00 | Exercício 10 | top-k sem ordenar tudo; quando usar cada estratégia |
| 8:00–9:30 | Exercício 11, rodando a célula | as razões normalizadas constantes; +2 operações por dobra no `insert` e +3 no `extract_max` |
| 9:30–10:00 | Exercício 12 | arquitetura em camadas e o teste integrado de 30.000 operações |

Dica para a gravação: rode **Executar tudo** antes de começar e explique a partir das saídas. Os números citados nas análises são os mesmos que aparecem nas células.